# AIC 2026 — Offline Pipeline Demo

Notebook này thử pipeline offline trên **một video thật** (`L21_V001`):

1. audit video và artifact có sẵn;
2. kiểm tra map-keyframe ↔ ảnh ↔ CLIP feature;
3. đọc object metadata;
4. tạo sampling plan stride 5 + mandatory anchors;
5. adaptive clustering trên CLIP feature;
6. redundancy elimination có bảo vệ anchor;
7. smoke-test nearest-neighbor index;
8. xuất artifact demo để bước query sử dụng sau.

> Notebook không thay đổi dữ liệu trong `archive/`. Mọi output được ghi vào `workspace/notebook_demo/`.

## 0. Kiểm tra kernel uv

Notebook được cấu hình để dùng kernel **AIC 2026 (uv)** từ `.venv`. Cell dưới đây xác nhận đúng Python executable trước khi chạy.

In [ ]:
import sys
print('Python executable:', sys.executable)
print('Kernel expected: AIC 2026 (uv)')
assert '.venv' in sys.executable, 'Hãy chọn kernel AIC 2026 (uv) trước khi chạy notebook.'

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ROOT = Path(r'C:\Users\ADMIN\AIC')
ARCHIVE = ROOT / 'archive'
OUTPUT = ROOT / 'workspace' / 'notebook_demo'
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('PADDLE_PDX_CACHE_HOME', str(ROOT / 'workspace' / 'paddlex_cache'))
os.environ.setdefault('PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK', 'True')

VIDEO_ID = 'L21_V001'
STRIDE = 5
RANDOM_STATE = 42

print('Output:', OUTPUT)

## 1. Tìm và audit artifact của video mẫu

In [ ]:
def find_exact(root: Path, filename: str) -> Path | None:
    matches = list(root.rglob(filename))
    if len(matches) > 1:
        print(f'Warning: có {len(matches)} file tên {filename}; dùng file đầu tiên')
    return matches[0] if matches else None

video_path = find_exact(ARCHIVE / 'video', f'{VIDEO_ID}.mp4')
keyframe_dir = next((p for p in (ARCHIVE / 'keyframes').rglob(VIDEO_ID) if p.is_dir()), None)
map_path = ARCHIVE / 'map-keyframes' / f'{VIDEO_ID}.csv'
clip_path = ARCHIVE / 'clip-features-32' / f'{VIDEO_ID}.npy'
metadata_path = ARCHIVE / 'media-info' / f'{VIDEO_ID}.json'
objects_dir = ARCHIVE / 'objects' / VIDEO_ID

paths = {
    'video': video_path,
    'keyframes': keyframe_dir,
    'map': map_path,
    'clip': clip_path,
    'metadata': metadata_path,
    'objects': objects_dir,
}

audit = {name: {'path': str(path) if path else None, 'exists': bool(path and path.exists())}
         for name, path in paths.items()}
pd.DataFrame(audit).T

## 2. Video metadata và original frame space

In [ ]:
assert video_path is not None, 'Không tìm thấy video mẫu'
cap = cv2.VideoCapture(str(video_path))
video_info = {
    'video_id': VIDEO_ID,
    'video_path': str(video_path),
    'fps': float(cap.get(cv2.CAP_PROP_FPS)),
    'frame_count': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
}
video_info['duration_seconds'] = video_info['frame_count'] / video_info['fps']
cap.release()

if metadata_path.exists():
    media_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    video_info['title'] = media_metadata.get('title')
    video_info['youtube_length'] = media_metadata.get('length')

pd.Series(video_info)

## 3. Kiểm tra map-keyframe, ảnh và CLIP feature

Đây là bước quan trọng nhất: số dòng map, số ảnh và số vector phải khớp nhau.

In [ ]:
frame_map = pd.read_csv(map_path).sort_values('n').reset_index(drop=True)
clip_features = np.load(clip_path).astype(np.float32)
image_paths = sorted(keyframe_dir.glob('*.jpg'), key=lambda p: int(p.stem))

checks = {
    'map_rows': len(frame_map),
    'image_count': len(image_paths),
    'clip_rows': clip_features.shape[0],
    'clip_dim': clip_features.shape[1],
    'frame_ids_monotonic': bool(frame_map.frame_idx.is_monotonic_increasing),
    'frame_ids_in_video': bool(frame_map.frame_idx.between(0, video_info['frame_count'] - 1).all()),
}
checks['counts_match'] = checks['map_rows'] == checks['image_count'] == checks['clip_rows']
assert checks['counts_match'], checks
assert checks['frame_ids_in_video'], checks

norms = np.linalg.norm(clip_features, axis=1, keepdims=True)
clip_features = clip_features / np.clip(norms, 1e-12, None)
frame_map['image_path'] = [str(p) for p in image_paths]
frame_map['clip_row'] = np.arange(len(frame_map))

pd.Series(checks)

In [ ]:
indices = np.linspace(0, len(frame_map) - 1, 8, dtype=int)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, idx in zip(axes.flat, indices):
    row = frame_map.iloc[idx]
    ax.imshow(Image.open(row.image_path).convert('RGB'))
    ax.set_title(f"n={int(row.n)} | frame={int(row.frame_idx)} | t={row.pts_time:.1f}s")
    ax.axis('off')
plt.tight_layout()

## 4. Chuẩn hóa object metadata

In [ ]:
SYNONYMS = {
    'man': 'person', 'woman': 'person', 'boy': 'person', 'girl': 'person',
    'mobile phone': 'phone', 'cell phone': 'phone',
    'motorbike': 'motorcycle',
}

def normalize_label(label: str) -> str:
    label = label.strip().lower()
    return SYNONYMS.get(label, label)

def parse_object_file(path: Path) -> list[dict]:
    data = json.loads(path.read_text(encoding='utf-8'))
    # Format thật của BTC dùng các array song song.
    if isinstance(data, dict) and 'detection_class_entities' in data:
        labels = data.get('detection_class_entities', [])
        scores = data.get('detection_scores', [])
        boxes = data.get('detection_boxes', [])
        detections = [
            {'label': label, 'confidence': float(scores[i]), 'bbox': boxes[i]}
            for i, label in enumerate(labels)
            if i < len(scores) and i < len(boxes)
        ]
    elif isinstance(data, list):
        detections = data
    else:
        detections = data.get('objects', data.get('detections', data.get('result', [])))
    rows = []
    for det in detections:
        label = str(det.get('class_name', det.get('label', det.get('class', 'unknown'))))
        confidence = det.get('score', det.get('confidence', det.get('probability')))
        rows.append({
            'label_original': label,
            'label_normalized': normalize_label(label),
            'confidence': confidence,
            'bbox': det.get('bbox', det.get('box')),
        })
    return rows

sample_object_files = sorted(objects_dir.glob('*.json'), key=lambda p: int(p.stem))[:10]
object_rows = []
for path in sample_object_files:
    keyframe_n = int(path.stem)
    match = frame_map.loc[frame_map.n == keyframe_n]
    frame_id = int(match.iloc[0].frame_idx) if len(match) else None
    for obj in parse_object_file(path):
        object_rows.append({'video_id': VIDEO_ID, 'frame_id': frame_id, **obj})

objects_sample = pd.DataFrame(object_rows)
objects_sample.head(20)

## 5. Sampling plan stride 5 + mandatory anchors

Trong demo chưa chạy OmniShotCut. Toàn video được xem như một vùng để kiểm tra logic sampling. Khi có shot detector, chỉ cần chạy cùng hàm cho từng `[start_frame, end_frame]`.

In [ ]:
def make_sampling_plan(start_frame: int, end_frame: int, stride: int = 5) -> pd.DataFrame:
    reasons: dict[int, set[str]] = {}

    def add(frame_id: int, reason: str):
        frame_id = max(start_frame, min(end_frame, int(frame_id)))
        reasons.setdefault(frame_id, set()).add(reason)

    for frame_id in range(start_frame, end_frame + 1, stride):
        add(frame_id, f'stride_{stride}')

    add(start_frame, 'shot_start')
    add((start_frame + end_frame) // 2, 'shot_middle')
    add(end_frame, 'shot_end')
    for delta in (-2, -1, 1, 2):
        add(start_frame + delta, 'boundary')
        add(end_frame + delta, 'boundary')

    rows = []
    for frame_id, frame_reasons in sorted(reasons.items()):
        is_mandatory = any(r != f'stride_{stride}' for r in frame_reasons)
        rows.append({
            'video_id': VIDEO_ID,
            'shot_id': 0,
            'frame_id': frame_id,
            'pts_time': frame_id / video_info['fps'],
            'sampling_reason': '|'.join(sorted(frame_reasons)),
            'is_mandatory': is_mandatory,
        })
    return pd.DataFrame(rows)

sampling_plan = make_sampling_plan(0, video_info['frame_count'] - 1, STRIDE)
print(f"Original frames: {video_info['frame_count']:,}")
print(f"Sampled frames:  {len(sampling_plan):,}")
print(f"Compression:     {1 - len(sampling_plan) / video_info['frame_count']:.2%}")
sampling_plan.head(10)

## 6. Adaptive clustering trên provided CLIP keyframes

Demo chọn `k` bằng silhouette score, giới hạn `k ≤ 12`. Trên pipeline đầy đủ, clustering được thực hiện riêng trong từng shot.

In [ ]:
def adaptive_cluster(features: np.ndarray, max_clusters: int = 12):
    n = len(features)
    if n < 3:
        labels = np.arange(n)
        return {'k': n, 'labels': labels, 'model': None, 'scores': {}}

    k_max = min(max_clusters, max(2, math.ceil(math.sqrt(n))), n - 1)
    candidates = {}
    for k in range(2, k_max + 1):
        model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
        labels = model.fit_predict(features)
        score = silhouette_score(features, labels, metric='cosine')
        candidates[k] = {'score': float(score), 'model': model, 'labels': labels}

    best_k = max(candidates, key=lambda k: candidates[k]['score'])
    best = candidates[best_k]
    return {
        'k': best_k,
        'labels': best['labels'],
        'model': best['model'],
        'scores': {k: v['score'] for k, v in candidates.items()},
    }

cluster_result = adaptive_cluster(clip_features)
print('Selected k:', cluster_result['k'])
pd.Series(cluster_result['scores'], name='silhouette_score')

In [ ]:
def select_medoids(features: np.ndarray, labels: np.ndarray, centers: np.ndarray) -> list[int]:
    medoids = []
    for cluster_id in sorted(np.unique(labels)):
        members = np.flatnonzero(labels == cluster_id)
        center = centers[cluster_id]
        center = center / max(np.linalg.norm(center), 1e-12)
        similarities = features[members] @ center
        medoids.append(int(members[np.argmax(similarities)]))
    return medoids

medoid_rows = select_medoids(
    clip_features, cluster_result['labels'], cluster_result['model'].cluster_centers_
)

# Anchor gần nhất trong provided-keyframe space.
target_anchor_frames = [
    int(frame_map.frame_idx.min()),
    int((frame_map.frame_idx.min() + frame_map.frame_idx.max()) // 2),
    int(frame_map.frame_idx.max()),
]
anchor_rows = [int((frame_map.frame_idx - target).abs().idxmin()) for target in target_anchor_frames]
candidate_rows = sorted(set(medoid_rows + anchor_rows))

candidates = frame_map.iloc[candidate_rows].copy()
candidates['is_mandatory'] = [row in anchor_rows for row in candidate_rows]
candidates['selection_reason'] = [
    'medoid+anchor' if row in medoid_rows and row in anchor_rows else
    'anchor' if row in anchor_rows else 'medoid'
    for row in candidate_rows
]
candidates[['n', 'frame_idx', 'pts_time', 'is_mandatory', 'selection_reason']]

## 7. Redundancy elimination có bảo vệ mandatory anchor

In [ ]:
def eliminate_redundancy(candidate_rows: list[int], mandatory_rows: set[int],
                           features: np.ndarray, threshold: float = 0.98):
    kept: list[int] = []
    decisions = []

    # Mandatory frames được đưa vào trước và không bao giờ bị loại.
    for row in sorted(mandatory_rows):
        if row in candidate_rows and row not in kept:
            kept.append(row)
            decisions.append({'row': row, 'kept': True, 'reason': 'mandatory', 'duplicate_of': None})

    for row in candidate_rows:
        if row in mandatory_rows:
            continue
        if not kept:
            kept.append(row)
            decisions.append({'row': row, 'kept': True, 'reason': 'first', 'duplicate_of': None})
            continue

        similarities = features[kept] @ features[row]
        best_pos = int(np.argmax(similarities))
        best_similarity = float(similarities[best_pos])
        duplicate_of = kept[best_pos]
        if best_similarity >= threshold:
            decisions.append({
                'row': row, 'kept': False, 'reason': 'visual_duplicate',
                'duplicate_of': duplicate_of, 'similarity': best_similarity,
            })
        else:
            kept.append(row)
            decisions.append({
                'row': row, 'kept': True, 'reason': 'distinct',
                'duplicate_of': None, 'similarity': best_similarity,
            })

    return sorted(kept), pd.DataFrame(decisions)

compact_rows, redundancy_log = eliminate_redundancy(
    candidate_rows, set(anchor_rows), clip_features, threshold=0.98
)
compact = frame_map.iloc[compact_rows].copy()
print(f'Candidates trước redundancy: {len(candidate_rows)}')
print(f'Compact keyframes:           {len(compact_rows)}')
redundancy_log

## 8. Smoke test nearest-neighbor index

Cell này dùng NumPy exact cosine search, chưa cần FAISS. Một keyframe được dùng làm query ảnh để xác nhận index và row mapping hoạt động đúng.

In [ ]:
def search_by_vector(query_vector: np.ndarray, matrix: np.ndarray, top_k: int = 8):
    query_vector = query_vector / max(np.linalg.norm(query_vector), 1e-12)
    scores = matrix @ query_vector
    top_rows = np.argsort(-scores)[:top_k]
    return top_rows, scores[top_rows]

query_row = len(frame_map) // 2
top_rows, top_scores = search_by_vector(clip_features[query_row], clip_features, top_k=8)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, row, score in zip(axes.flat, top_rows, top_scores):
    item = frame_map.iloc[row]
    ax.imshow(Image.open(item.image_path).convert('RGB'))
    ax.set_title(f"frame={int(item.frame_idx)} | score={score:.3f}")
    ax.axis('off')
plt.tight_layout()

## 9. OCR tùy chọn

PaddleOCR và model tiếng Việt đã được cài trong môi trường uv. `enable_mkldnn=False` được dùng để tránh lỗi oneDNN của PaddlePaddle trên Windows CPU.

In [ ]:
RUN_OCR = True
OCR_LIMIT = 3

ocr_rows = []
if RUN_OCR:
    from paddleocr import PaddleOCR

    ocr = PaddleOCR(
        lang='vi',
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        enable_mkldnn=False,
    )
    for row in frame_map.head(OCR_LIMIT).itertuples():
        result = ocr.predict(row.image_path)
        ocr_rows.append({
            'video_id': VIDEO_ID,
            'frame_id': int(row.frame_idx),
            'raw_result': str(result),
        })
    display(pd.DataFrame(ocr_rows))
else:
    print('OCR đang tắt; đặt RUN_OCR=True sau khi cài PaddleOCR.')

## 10. Xuất artifact demo

In [ ]:
frame_manifest_path = OUTPUT / f'{VIDEO_ID}_provided_frames.csv'
sampling_plan_path = OUTPUT / f'{VIDEO_ID}_stride5_plan.csv'
features_path = OUTPUT / f'{VIDEO_ID}_clip_normalized.npy'
compact_path = OUTPUT / f'{VIDEO_ID}_compact_keyframes.csv'
redundancy_path = OUTPUT / f'{VIDEO_ID}_redundancy_log.csv'
report_path = OUTPUT / f'{VIDEO_ID}_report.json'

frame_map.to_csv(frame_manifest_path, index=False, encoding='utf-8-sig')
sampling_plan.to_csv(sampling_plan_path, index=False, encoding='utf-8-sig')
np.save(features_path, clip_features.astype(np.float16))
compact.to_csv(compact_path, index=False, encoding='utf-8-sig')
redundancy_log.to_csv(redundancy_path, index=False, encoding='utf-8-sig')

report = {
    'video': video_info,
    'audit': audit,
    'checks': checks,
    'stride': STRIDE,
    'planned_samples': len(sampling_plan),
    'adaptive_k': cluster_result['k'],
    'candidate_keyframes': len(candidate_rows),
    'compact_keyframes': len(compact_rows),
}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame({
    'artifact': ['frame_manifest', 'sampling_plan', 'features', 'compact', 'redundancy_log', 'report'],
    'path': [frame_manifest_path, sampling_plan_path, features_path, compact_path, redundancy_path, report_path],
})

## Bước tiếp theo

Sau khi notebook chạy thành công trên `L21_V001`:

1. thay pseudo-shot bằng output TransNetV2/OmniShotCut;
2. decode và embed các frame trong sampling plan stride 5;
3. chạy PaddleOCR theo batch có resume;
4. chuyển code notebook thành module trong `src/aic/`;
5. chạy thử 10 video trước khi xử lý toàn bộ 961 video;
6. xây FAISS index sau khi row mapping đã vượt qua integrity test.